In [4]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression

In [5]:
df = np.round(pd.read_csv('50_Startups.csv')[['R&D Spend','Administration','Marketing Spend','Profit']]/10000)
np.random.seed(9)
df = df.sample(5)
df

,R&D Spend,Administration,Marketing Spend,Profit
21,8.0,15.0,30.0,11.0
37,4.0,5.0,20.0,9.0
2,15.0,10.0,41.0,19.0
14,12.0,16.0,26.0,13.0
44,2.0,15.0,3.0,7.0


In [6]:
df = df.iloc[:,0:-1].copy()
df

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,4.0,5.0,20.0
2,15.0,10.0,41.0
14,12.0,16.0,26.0
44,2.0,15.0,3.0


In [7]:
df.iloc[1,0] = np.nan
df.iloc[3,1] = np.nan
df.iloc[-1,-1] = np.nan

In [8]:
df.head()

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,NaN,5.0,20.0
2,15.0,10.0,41.0
14,12.0,NaN,26.0
44,2.0,15.0,NaN


# Step 1 - Impute all missing values with mean of respective col


In [9]:
df0 = pd.DataFrame()

df0['R&D Spend'] = df['R&D Spend'].fillna(df['R&D Spend'].mean())
df0['Administration'] = df['Administration'].fillna(df['Administration'].mean())

# Oth iteration

In [10]:
df0

,R&D Spend,Administration
21,8.00,15.00
37,9.25,5.00
2,15.00,10.00
14,12.00,11.25
44,2.00,15.00


# Remove the col1 imputed value

In [11]:
df1 = df0.copy()

df1.iloc[1,0] = np.nan

df1

,R&D Spend,Administration
21,8.0,15.00
37,NaN,5.00
2,15.0,10.00
14,12.0,11.25
44,2.0,15.00


# Use first 3 rows to build a model and use the last for prediction

In [12]:

X = df1.iloc[[0,2,3,4],1:3]
X

,Administration
21,15.00
2,10.00
14,11.25
44,15.00


In [13]:
y = df1.iloc[[0,2,3,4],0]
y

,R&D Spend
21,8.0
2,15.0
14,12.0
44,2.0


In [14]:
lr = LinearRegression()
lr.fit(X,y)
prediction_input = pd.DataFrame(df1.iloc[1,1:].values.reshape(1,1), columns=['Administration'])
lr.predict(prediction_input)

array([24.56862745])

In [15]:
df1.iloc[1,0] = 23.14

In [16]:
df1

,R&D Spend,Administration
21,8.00,15.00
37,23.14,5.00
2,15.00,10.00
14,12.00,11.25
44,2.00,15.00


# Remove the col2 imputed value

In [17]:
df1.iloc[3,1] = np.nan

df1

,R&D Spend,Administration
21,8.00,15.0
37,23.14,5.0
2,15.00,10.0
14,12.00,NaN
44,2.00,15.0


# Use last 3 rows to build a model and use the first for prediction

In [18]:
X = df1.iloc[[0,1,2,4],[0,1]]
X

,R&D Spend,Administration
21,8.00,15.0
37,23.14,5.0
2,15.00,10.0
44,2.00,15.0


In [19]:
y = df1.iloc[[0,1,2,4],1]
y

,Administration
21,15.0
37,5.0
2,10.0
44,15.0


In [40]:
# Define X and y for training, ensuring no NaNs and consistent rows.
# Let's use 'R&D Spend' as the feature and 'Administration' as the target.
# We will use rows [0,1,2,4] from df1 as they do not have NaNs in both column 0 and 1.
X_train = df1.iloc[[0,1,2,4],0].to_frame()  # Feature: 'R&D Spend'
y_train = df1.iloc[[0,1,2,4],1]             # Target: 'Administration'

lr = LinearRegression()
lr.fit(X_train, y_train)

# Prepare the prediction input for the row with NaN (iloc index 3, original index 14)
# The feature for prediction is 'R&D Spend' (column 0) from this row.
prediction_feature_value = df1.iloc[3,0] # This is 12.0 for 'R&D Spend'

# Create a DataFrame for prediction input with the correct feature name, as the model was trained with feature names.
prediction_input_df = pd.DataFrame([prediction_feature_value], columns=X_train.columns)

lr.predict(prediction_input_df)

array([15.43103624])

In [22]:
y = df1.iloc[0:4,-1]
y

,Administration
21,15.0
37,5.0
2,10.0
14,NaN


In [26]:
# Define training data X and y, ensuring no NaNs and consistent rows
# Let's predict 'Administration' (column 1) based on 'R&D Spend' (column 0).
# We will use rows [0,1,2,4] from df1 as they do not have NaNs in either column 0 or 1 (original indices 21, 37, 2, 44).
X_train_corrected = df1.iloc[[0,1,2,4],0].to_frame() # R&D Spend as feature
y_train_corrected = df1.iloc[[0,1,2,4],1]           # Administration as target

lr = LinearRegression()
lr.fit(X_train_corrected, y_train_corrected)

# Prepare the prediction input
# If our model uses 'R&D Spend' as the feature, the prediction input should be the
# 'R&D Spend' value for the 5th row (iloc index 4, original index 44).
prediction_value_input = df1.iloc[4,0]

# Create a DataFrame for prediction input with the correct feature name
prediction_input_df = pd.DataFrame([prediction_value_input], columns=X_train_corrected.columns)

lr.predict(prediction_input_df)

array([16.32098555])

In [27]:
df1.iloc[4,-1] = 31.56

In [28]:
df1

,R&D Spend,Administration
21,8.00,15.00
37,23.14,5.00
2,15.00,10.00
14,12.00,NaN
44,2.00,31.56


# Subtract 0th iteration from 1st iteration

In [29]:
df1 - df0

,R&D Spend,Administration
21,0.00,0.00
37,13.89,0.00
2,0.00,0.00
14,0.00,NaN
44,0.00,16.56


In [31]:
df2 = df1.copy()

df2.iloc[1,0] = np.nan

df2

,R&D Spend,Administration
21,8.0,15.00
37,NaN,5.00
2,15.0,10.00
14,12.0,NaN
44,2.0,31.56


In [41]:
admin_mean = df2['Administration'].dropna().mean()
df2.iloc[3,1] = admin_mean

X = df2.iloc[[0,2,3,4],1:2] # Selecting 'Administration' as the feature
y = df2.iloc[[0,2,3,4],0]  # 'R&D Spend' as the target

lr = LinearRegression()
lr.fit(X,y)

# Prepare prediction input: 'Administration' value for df2.iloc[1] (original index 37)
# X was trained with a single column DataFrame, so predict input should match
prediction_admin_value = df2.iloc[1,1] # This is 5.0
prediction_input_df = pd.DataFrame([prediction_admin_value], columns=X.columns)

lr.predict(prediction_input_df)

array([16.38953464])

In [33]:
df2.iloc[1,0] = 23.78

In [49]:
# The NaN at df2.iloc[3,1] is intentionally introduced here to be imputed later by prediction.
df2.iloc[3,1] = np.nan

# For training the model, we need to select rows where both 'R&D Spend' (feature)
# and 'Administration' (target) are not NaN.
# Also, we should exclude the row we intend to predict (iloc index 3).
# Based on the current state of df2:
# df2.iloc[0] (original index 21): [8.0, 15.0] -> clean
# df2.iloc[1] (original index 37): [23.78, 5.0] -> clean
# df2.iloc[2] (original index 2): [15.0, 10.0] -> clean
# df2.iloc[3] (original index 14): [12.0, NaN] -> this is the target row for prediction
# df2.iloc[4] (original index 44): [2.0, NaN] -> has NaN in target, cannot be used for training y

# So, we will use rows [0, 1, 2] for training.
X_train = df2.iloc[[0,1,2], 0].to_frame()  # Feature: 'R&D Spend'
y_train = df2.iloc[[0,1,2], 1]             # Target: 'Administration'

lr = LinearRegression()
lr.fit(X_train, y_train)

# Prepare the prediction input for the row with NaN (iloc index 3, original index 14)
# The feature for prediction is 'R&D Spend' (column 0) from this row.
prediction_feature_value = df2.iloc[3,0] # This is 12.0 for 'R&D Spend'

# Create a DataFrame for prediction input with the correct feature name,
# as the model was trained with feature names.
prediction_input_df = pd.DataFrame([prediction_feature_value], columns=X_train.columns)

lr.predict(prediction_input_df)

array([12.26752668])

In [37]:
df2.iloc[3,1] = 11.22

In [51]:
# The NaN at df2.iloc[4,-1] is intentionally introduced here to be imputed later by prediction.
df2.iloc[4,-1] = np.nan

# For training the model, we need to select rows where both 'R&D Spend' (feature)
# and 'Administration' (target) are not NaN.
# We exclude df2.iloc[4] from training, as its 'Administration' value is NaN and is our prediction target.
# Based on the current state of df2:
# df2.iloc[0] (original index 21): [8.0, 15.0] -> clean
# df2.iloc[1] (original index 37): [23.78, 5.0] -> clean
# df2.iloc[2] (original index 2): [15.0, 10.0] -> clean
# df2.iloc[3] (original index 14): [12.0, NaN] -> this row has NaN in target, cannot be used for training y
# df2.iloc[4] (original index 44): [2.0, NaN] -> this is the target row for prediction

# So, we will use rows [0, 1, 2] for training.
X_train = df2.iloc[[0,1,2],0].to_frame()  # Feature: 'R&D Spend' from rows 0, 1, 2
y_train = df2.iloc[[0,1,2],1]             # Target: 'Administration' from rows 0, 1, 2

lr = LinearRegression()
lr.fit(X_train, y_train)

# Prepare the prediction input for the row with NaN (iloc index 4, original index 44)
# The feature for prediction is 'R&D Spend' (column 0) from this row.
prediction_feature_value = df2.iloc[4,0] # This is 2.0 for 'R&D Spend'

# Create a DataFrame for prediction input with the correct feature name,
# as the model was trained with feature names.
prediction_input_df = pd.DataFrame([prediction_feature_value], columns=X_train.columns)

lr.predict(prediction_input_df)

ValueError: Input y contains NaN.

In [39]:
df2.iloc[4,-1] = 31.56

In [43]:
df2

,R&D Spend,Administration
21,8.00,15.000
37,23.78,5.000
2,15.00,10.000
14,12.00,14.556
44,2.00,NaN


In [44]:
df2 - df1

,R&D Spend,Administration
21,0.00,0.0
37,0.64,0.0
2,0.00,0.0
14,0.00,NaN
44,0.00,NaN


In [48]:
df3 = df2.copy()

df3.iloc[1,0] = np.nan

df3

,R&D Spend,Administration
21,8.0,15.0
37,NaN,5.0
2,15.0,10.0
14,12.0,NaN
44,2.0,NaN


In [50]:
X = df3.iloc[[0,2,3,4],1:3]
y = df3.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[1,1:].values.reshape(1,2))

ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values